In [ ]:

import sys

if "google.colab" in sys.modules:
    # If running in Google Colab
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Install dependancies
    !pip install pymatgen torch_geometric tqdm

    # Set project path
    csv_data = "/content/drive/MyDrive/DATA/Dataset.csv"
    cifs_path = "/content/drive/MyDrive/DATA/"

else:
    csv_data = "DATA/Dataset.csv"
    cifs_path = "DATA/"

Mounted at /content/drive


In [ ]:
# Imports
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN

from sklearn.preprocessing import StandardScaler

In [ ]:
# Load CSV dataset
df = pd.read_csv(csv_data)

df.head()


In [ ]:
print("Total samples:", len(df))

# Prepare tabular data

In [ ]:
# Check class balance
print("\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
#Drop leakage columns
df = df.drop(columns=[
    "mp_material_id",
    "mp_formula",
    "cif"
], errors="ignore")

In [ ]:
# Extract other features

def extract_pymatgen_features(cif_file):
    structure = Structure.from_file(cifs_path + cif_file)

    # Basic structural properties
    volume = structure.volume
    density = structure.density
    num_atoms = len(structure)

    # Atomic properties
    elements = [site.specie for site in structure]

    for_mean_mass = []

    for el in elements:
        try:
            for_mean_mass.append(el.atomic_mass)
        except:
            for_mean_mass.append(0)

    mean_mass = np.mean(for_mean_mass)
    mean_eneg = np.mean([el.X for el in elements if el.X is not None])

    add_features = [volume, density, num_atoms, mean_mass, mean_eneg]

    return add_features
class GaussianDistance:
    def __init__(self, dmin=0, dmax=6, step=0.15):
        self.filter = np.arange(dmin, dmax + step, step)
        self.var = step

    def expand(self, distances):
        return np.exp(-((distances[..., np.newaxis] - self.filter) ** 2) / (self.var ** 2))

gaussian_expansion = GaussianDistance()

In [ ]:
# Apply feature extraction to all samples
additional_features = []

for cif in df['cif']:
    features = extract_pymatgen_features(cif)
    additional_features.append(features)

additional_features_df = pd.DataFrame(additional_features, columns=[
    'volume', 'density', 'num_atoms', 'mean_mass', 'mean_eneg'
])

# Combine with original dataframe
df = pd.concat([df, additional_features_df], axis=1)

# Prepare graphical data

In [ ]:
# Atomic feature extractor
from pymatgen.core.periodic_table import Element

def get_node_features(structure):
    features = []

    for site in structure:
        el = Element(site.specie.symbol)

        feat = [
            el.Z,
            el.X if el.X else 0,
            el.atomic_radius if el.atomic_radius else 0,
            el.atomic_mass,
            el.group if el.group else 0,
            el.row,
            int(el.is_metal),
            el.mendeleev_no
        ]

        features.append(feat)

    node_features_all = np.array(features, dtype=float)

    node_features_all = np.vstack(node_features_all)

    node_scaler = StandardScaler()
    node_scaler.fit(node_features_all)

In [ ]:
class GaussianDistance:
    def __init__(self, dmin=0, dmax=6, step=0.15):
        self.filter = np.arange(dmin, dmax + step, step)
        self.var = step

    def expand(self, distances):
        return np.exp(-((distances[..., np.newaxis] - self.filter) ** 2) / (self.var ** 2))

gaussian_expansion = GaussianDistance()

'''# Gaussian Distance Expansion
class GaussianExpansion:
    def __init__(self, dmin=0, dmax=6, step=0.2):
        self.centers = np.arange(dmin, dmax+step, step)
        self.width = step

    def expand(self, distance):
        return np.exp(-((distance - self.centers) ** 2) / self.width**2)

gaussian = GaussianExpansion()'''


def get_edges(structure):
    edge_index = []
    edge_attr = []

    cutoff = 6.0

    for i, site in enumerate(structure):
        neighbors = structure.get_neighbors(site, cutoff)

        for n in neighbors:
            j = n.index
            dist = n.nn_distance

            edge_index.append([i, j])
            edge_attr.append(dist)


    edge_attr = np.array(edge_attr)
    edge_attr = gaussian_expansion.expand(edge_attr)

    return edge_index, edge_attr

In [ ]:
def get_global_features(row, structure):
    # Extract from df

    GLOBAL_FEATURES = [
    'spacegroup_number',   # integer space group (1–230)
    'number of atoms',     # atoms per unit cell
    'Band Gap',            # band gap in eV
    'a', 'b', 'c',         # lattice constants (Angstroms)
    'alpha', 'beta', 'gamma',  # lattice angles (degrees)
    'Z',                   # mean atomic number
    'electronegativity'    # mean Pauling electronegativity
]

    globals = [row[feature] for feature in GLOBAL_FEATURES]

    # Extracted from structure
    add_globals = [
        structure.volume,
        structure.density,
        len(structure.composition.elements),
        np.mean([site.specie.atomic_mass for site in structure]),
        np.mean([site.specie.X for site in structure if site.specie.X is not None])
    ]

    return globals + add_globals

In [ ]:
def get_graph(row):
    cif_file = row['cif']

    # The struucture
    structure = Structure.from_file(cifs_path + cif_file)

    node_features = get_node_features(structure)
    edge_index, edge_attr = get_edges(structure)
    global_features = get_global_features(row, structure)
    label = row["label"]

    return Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        u = torch.tensor(global_features, dtype=torch.float),
        y=torch.tensor([label], dtype=torch.long)
    )

In [ ]:
# Create graphs for a whole dataset

def data_graph(df):
    # run get_graph for each row in the dataframe
    graphs = []
    for _, row in df.iterrows():
        graph = get_graph(row)
        graphs.append(graph)
    return graphs

100%|██████████| 7000/7000 [1:09:00<00:00,  1.69it/s]


Graphs saved: 7000


In [ ]:



def build_graph(row, cutoff=5.0):
    """
    Build a PyG Data object from one row of df_model.
    row: pandas Series containing at least 'cif' and global feature columns.
    """
    # Construct full CIF path (CSV stores relative path
    cif_path = os.path.join(CIF_DIR, row['cif'])
    parser = CifParser(cif_path)
    structure = parser.get_structures()[0]

    # Node features
    nodes = []

    for site in structure:
        node_features = [
            site.specie.Z if site.specie.Z is not None else 0,                                       # Atomic number of the atom
            site.specie.X if site.specie.X is not None else 0.0, # Electronegativity of the atom; set to 0.0 if missing
            float(site.specie.atomic_mass) if site.specie.atomic_mass is not None else 0.0,                      # Atomic mass of the atom
            site.specie.row if site.specie.row is not None else 0.0,                                     # Row number of atom in periodic table
            site.specie.group if site.specie.group is not None else 0.0,                                   # Group number of the atom
            site.specie.atomic_radius if site.specie.atomic_radius is not None else 0.0,                           # Atom's atomic radius
            int(site.specie.is_metal) if site.specie.is_metal is not None else 0,                           # If element is a metal
            site.specie.mendeleev_no if site.specie.mendeleev_no is not None else 0.0                            # Chemical ordering number
        ]
        nodes.append(node_features)
        # Example: compute basic composition statistics via pymatgen
from pymatgen.core import Element
elements = []
for sym in df['mp_formula']:
    for el, amt in Composition(sym).items():
        elements.extend([el] * amt)
df['mean_atomic_radius'] = [pd.Series([Element(el).atomic_radius for el in elist]).mean() for elist in element_lists]
df['std_atomic_radius']  = [pd.Series([Element(el).atomic_radius for el in elist]).std()  for elist in element_lists]
# ... similarly compute mean_atomic_mass, etc.



    x = torch.tensor(nodes, dtype=torch.float)   # [N, 2]

    # Atomic positions
    pos = torch.tensor(structure.cart_coords, dtype=torch.float)      # [N, 3]

    # Edges via radius graph
    edge_index = radius_graph(pos, r=cutoff, loop=False)              # [2, E]
    src, dst = edge_index
    edge_attr = torch.norm(pos[src] - pos[dst], dim=1, keepdim=True) # [E, 1]

    #  Global features
    vals = [float(row[col]) for col in GLOBAL_FEATURES]
    vals.append(structure.volume)
    vals.append(structure.density)
    vals.append(len(structure.composition.elements))

    u = torch.tensor(vals, dtype=torch.float).unsqueeze(0)

    #Label
    y = torch.tensor([row['label']], dtype=torch.long)               # [1]

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                u=u, y=y, pos=pos)
    return data

# Quick test
sample = build_graph(train_df.iloc[0])
print("Sample graph:", sample)
print("Node features shape:", sample.x.shape)
print("Edge index shape:", sample.edge_index.shape)
print("Edge attributes shape:", sample.edge_attr.shape)
print("Global features shape:", sample.u.shape)
print("Label:", sample.y.item())

In [ ]:
# Create PyG Dataset classes
class TopologicalInsulatorDataset(Dataset):
    def __init__(self, dataframe):
        super().__init__()
        self.df = dataframe.reset_index(drop=True)

    def len(self):
        return len(self.df)

    def get(self, idx):
        return build_graph(self.df.iloc[idx])

# Instantiate datasets
train_dataset = TopologicalInsulatorDataset(train_df)
val_dataset   = TopologicalInsulatorDataset(val_df)
test_dataset  = TopologicalInsulatorDataset(test_df)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")



In [ ]:
# Graph Construction Function

CUTOFF = 6.0  # Angstrom

def build_graph(row):

    structure = Structure.from_file(os.path.join(PROJECT_PATH, row['cif']))

    # -------------------------
    # 🔹 Node features
    # -------------------------
    node_features = []

    for site in structure:
        el = site.specie

        node_features.append([
            el.Z,
            el.X if el.X else 0,
            el.atomic_radius if el.atomic_radius else 0,
            el.nvalence() if hasattr(el, "nvalence") else 0,
            el.ionization_energy if el.ionization_energy else 0,
            el.electron_affinity if el.electron_affinity else 0,
            el.row,
            el.group
        ])

    x = torch.tensor(node_features, dtype=torch.float)

    # Positions
    pos = torch.tensor(structure.cart_coords, dtype=torch.float)

    # Edge construction
    edge_index = radius_graph(pos, r=CUTOFF, loop=False)

    src, dst = edge_index

    distances = torch.norm(pos[src] - pos[dst], dim=1)

    # Gaussian expansion
    centers = torch.linspace(0, CUTOFF, 10)

    edge_attr = torch.exp(-((distances.unsqueeze(1) - centers)**2))

    # Global features
    u = torch.tensor(row[global_cols].astype(float).values,
                     dtype=torch.float).unsqueeze(0)

    # Label
    y = torch.tensor([row['label']], dtype=torch.long)

    return Data(x=x, edge_index=edge_index,
                edge_attr=edge_attr, pos=pos,
                u=u, y=y)
# Precompute graphs

def save_graphs(df, split_name):

    graphs = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            g = build_graph(row)
            graphs.append(g)
        except:
            continue

    torch.save(graphs, os.path.join(PROJECT_PATH, f"{split_name}.pt"))
    print(f"{split_name} saved: {len(graphs)} graphs")

# Run once
save_graphs(train_df, "train")
save_graphs(val_df, "val")
save_graphs(test_df, "test")

In [ ]:
# =========================================================
# Gaussian basis expansion (edge features)
# =========================================================
N_GAUSS = 40
SIGMA   = 0.5
centers = torch.linspace(0, 6.0, N_GAUSS)

def gaussian_expand(d):
    return torch.exp(-((d - centers) ** 2) / (SIGMA ** 2))
# =========================================================
# Node feature extraction
# =========================================================
def get_node_features(site):
    el = site.specie
    return [
        el.Z or 0,
        el.X or 0,
        el.atomic_mass or 0,
        el.row or 0,
        el.group or 0,
        el.atomic_radius or 0,
        int(el.is_metal),
        el.mendeleev_no or 0,
    ]

NODE_DIM = 8
# =========================================================
# Fit node + global feature scalers
# =========================================================
node_feats = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    try:
        struct = CifParser(os.path.join(CIF_DIR, row['cif'])).get_structures()[0]
        for site in struct:
            node_feats.append(get_node_features(site))
    except:
        pass

node_scaler = StandardScaler()
node_scaler.fit(node_feats)

GLOBAL_COLS = [
    'spacegroup_number', 'number of atoms', 'Band Gap',
    'a','b','c','alpha','beta','gamma','Z','electronegativity'
]

global_scaler = StandardScaler()
global_scaler.fit(train_df[GLOBAL_COLS])
# =========================================================
# Build graph with periodic neighbors
# =========================================================
def build_graph(row):
    path = os.path.join(CIF_DIR, row['cif'])
    struct = CifParser(path).get_structures()[0]

    # Node features
    feats = [get_node_features(s) for s in struct]
    x = torch.tensor(node_scaler.transform(feats), dtype=torch.float)

    # PBC neighbors
    edge_src, edge_dst, dist_list, vec_list = [], [], [], []

    all_nbrs = struct.get_all_neighbors(6.0)

    for i, nbrs in enumerate(all_nbrs):
        nbrs = sorted(nbrs, key=lambda x: x.nn_distance)[:12]

        for nbr in nbrs:
            edge_src.append(i)
            edge_dst.append(nbr.index)

            d = nbr.nn_distance
            vec = nbr.coords - struct[i].coords

            dist_list.append([d])
            vec_list.append(vec)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    dist = torch.tensor(dist_list, dtype=torch.float)
    vec  = torch.tensor(vec_list, dtype=torch.float)

    edge_attr = torch.cat([gaussian_expand(dist), vec/6.0], dim=1)

    # Global features
    g = global_scaler.transform([row[GLOBAL_COLS]])[0]
    extra = [struct.volume, struct.density, len(struct.species)]
    u = torch.tensor(list(g) + extra, dtype=torch.float).unsqueeze(0)

    y = torch.tensor([int(row['label'])])

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, y=y)
# =========================================================
# Build and save graphs
# =========================================================
def save_graphs(df_split):
    paths = []
    for _, row in tqdm(df_split.iterrows(), total=len(df_split)):
        p = os.path.join(GRAPH_DIR, row['mp_material_id'] + '.pt')
        if not os.path.exists(p):
            try:
                g = build_graph(row)
                torch.save(g, p)
            except:
                continue
        paths.append(p)
    return paths

train_paths = save_graphs(train_df)
val_paths   = save_graphs(val_df)
test_paths  = save_graphs(test_df)
class CrystalDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def len(self): return len(self.paths)
    def get(self, idx): return torch.load(self.paths[idx])

train_loader = DataLoader(CrystalDataset(train_paths), batch_size=32, shuffle=True)
val_loader   = DataLoader(CrystalDataset(val_paths), batch_size=32)
test_loader  = DataLoader(CrystalDataset(test_paths), batch_size=32)

### Split graphical data

In [ ]:

labels = [g.y.item() for g in graphs]

train_idx, temp_idx = train_test_split(
    range(len(graphs)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=[labels[i] for i in temp_idx]
)

train_set = [graphs[i] for i in train_idx]
val_set = [graphs[i] for i in val_idx]
test_set = [graphs[i] for i in test_idx]

# Save the split data
torch.save(train_set, os.path.join(PROJECT_PATH, "train_set.pt"))
torch.save(val_set, os.path.join(PROJECT_PATH, "val_set.pt"))
torch.save(test_set, os.path.join(PROJECT_PATH, "test_set.pt"))

In [9]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [10]:
# CharlesCGCNN Model
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool

class CharlesCGCNN(nn.Module):
    def __init__(self, node_dim, edge_dim):
        super().__init__()

        self.conv1 = GCNConv(node_dim, 128)
        self.conv2 = GCNConv(128, 128)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()

        x = global_mean_pool(x, batch)
        return self.fc(x)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CharlesCGCNN(node_dim=5, edge_dim=len(gaussian.centers)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [12]:
def train_epoch(loader):
    model.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
def evaluate(loader):
    model.eval()

    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)

            prob = torch.softmax(out, dim=1)[:,1]

            y_true += data.y.cpu().numpy().tolist()
            y_pred += out.argmax(dim=1).cpu().numpy().tolist()
            y_prob += prob.cpu().numpy().tolist()

    print(classification_report(y_true, y_pred, digits=3))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))

In [14]:
for epoch in range(30):
    loss = train_epoch(train_loader)
    print(f"Epoch {epoch}: Loss = {loss:.4f}")

Epoch 0: Loss = 0.6397
Epoch 1: Loss = 0.6221
Epoch 2: Loss = 0.6124
Epoch 3: Loss = 0.6038
Epoch 4: Loss = 0.5908
Epoch 5: Loss = 0.5830
Epoch 6: Loss = 0.5784
Epoch 7: Loss = 0.5756
Epoch 8: Loss = 0.5696
Epoch 9: Loss = 0.5657
Epoch 10: Loss = 0.5598
Epoch 11: Loss = 0.5500
Epoch 12: Loss = 0.5513
Epoch 13: Loss = 0.5414
Epoch 14: Loss = 0.5406
Epoch 15: Loss = 0.5311
Epoch 16: Loss = 0.5321
Epoch 17: Loss = 0.5240
Epoch 18: Loss = 0.5278
Epoch 19: Loss = 0.5180
Epoch 20: Loss = 0.5091
Epoch 21: Loss = 0.5112
Epoch 22: Loss = 0.5072
Epoch 23: Loss = 0.5002
Epoch 24: Loss = 0.5025
Epoch 25: Loss = 0.4951
Epoch 26: Loss = 0.5030
Epoch 27: Loss = 0.4894
Epoch 28: Loss = 0.4806
Epoch 29: Loss = 0.4858


In [15]:
print("Train Performance:")
evaluate(train_loader)

print("Validation Performance:")
evaluate(val_loader)

print("Test Performance:")
evaluate(test_loader)

Train Performance:
              precision    recall  f1-score   support

           0      0.840     0.628     0.719      2475
           1      0.698     0.878     0.778      2425

    accuracy                          0.752      4900
   macro avg      0.769     0.753     0.748      4900
weighted avg      0.770     0.752     0.748      4900

ROC-AUC: 0.8583740914297615
Validation Performance:
              precision    recall  f1-score   support

           0      0.826     0.597     0.693       531
           1      0.679     0.871     0.763       519

    accuracy                          0.732      1050
   macro avg      0.752     0.734     0.728      1050
weighted avg      0.753     0.732     0.727      1050

ROC-AUC: 0.8379978881595418
Test Performance:
              precision    recall  f1-score   support

           0      0.810     0.613     0.698       530
           1      0.684     0.854     0.760       520

    accuracy                          0.732      1050
   macro av

In [16]:
# Attention-based CharlesCGCNN
import torch.nn as nn
from torch_geometric.nn import GATConv, global_mean_pool

class CharlesCGCNN_Attention(nn.Module):
    def __init__(self, node_dim):
        super().__init__()

        # Attention layers
        self.gat1 = GATConv(node_dim, 128, heads=4, concat=True)
        self.gat2 = GATConv(128*4, 128, heads=1, concat=True)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, data, return_attention=False):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # First attention layer
        x, attn1 = self.gat1(x, edge_index, return_attention_weights=True)
        x = x.relu()

        # Second attention layer
        x, attn2 = self.gat2(x, edge_index, return_attention_weights=True)
        x = x.relu()

        pooled = global_mean_pool(x, batch)
        out = self.fc(pooled)

        if return_attention:
            return out, (attn1, attn2)
        return out

In [17]:
model = CharlesCGCNN_Attention(node_dim=5).to(device)

In [18]:
def get_attention_sample(data):
    model.eval()
    data = data.to(device)

    out, (attn1, attn2) = model(data, return_attention=True)

    return {
        "prediction": out.argmax(dim=1).item(),
        "prob": torch.softmax(out, dim=1)[0,1].item(),
        "attn1": attn1,
        "attn2": attn2
    }

In [19]:
def node_saliency(data):
    model.eval()
    data = data.to(device)

    data.x.requires_grad = True

    out = model(data)
    pred = out[:,1].sum()

    pred.backward()

    saliency = data.x.grad.abs().sum(dim=1).detach().cpu().numpy()

    return saliency

In [20]:
sample = test_set[0]

# Attention
attn_data = get_attention_sample(sample)

print("Prediction:", attn_data["prediction"])
print("Topological Probability:", attn_data["prob"])

# Plot attention
plot_attention_distribution(attn_data["attn2"])

# Node importance
saliency = node_saliency(sample)
plot_node_importance(saliency)

Prediction: 0
Topological Probability: 0.32237371802330017


NameError: name 'plot_attention_distribution' is not defined